# DataScienceJobs — Silver Layer Insights

This notebook focuses on **Business Intelligence and Market Trends** using the cleaned and enriched Silver data.
We ignore the raw 'Bronze' noise here and look at the actual state of the Canadian Data Science market.

In [1]:
import sys, os
from pathlib import Path

# ── Set working directory to project root ───────────────────────────────────
root_dir = Path(os.getcwd()).parent
os.chdir(root_dir)
sys.path.append(str(root_dir))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pipeline.config import SUPABASE_URL, SUPABASE_SERVICE_KEY
from supabase import create_client

client = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)

# ── Load Enriched Silver Data ────────────────────────────────────────────────
resp = client.table("job_postings").select(
    "title, company_name, location_city, location_state, is_remote, "
    "employment_type, salary_min, salary_max, salary_period, "
    "skills_tags, seniority, years_experience_min, posted_at"
).execute()

df = pd.DataFrame(resp.data)
df["posted_at"] = pd.to_datetime(df["posted_at"], utc=True, errors="coerce")

print(f"Analyzing {len(df):,} Cleaned Job Postings")

Analyzing 645 Cleaned Job Postings


## 1. Salary Distribution by Seniority
Examine the annual CAD salary ranges across different career levels.

In [2]:
salary_df = df[df["salary_min"].notna() & (df["salary_period"] == "YEAR")].copy()

fig = px.box(
    salary_df, 
    x="seniority", 
    y="salary_min",
    color="seniority",
    points="all",
    title="Annual Min Salary Distribution by Seniority (CAD)",
    category_orders={"seniority": ["Intern", "Junior", "Mid", "Senior", "Lead", "Director+"]},
    template="plotly_dark"
)
fig.show()

## 2. Skill Heatmap: Seniority vs Technology
Which skills are most demanded at each seniority level?

In [3]:
exploded = df.explode("skills_tags")
top_skills = exploded["skills_tags"].value_counts().head(15).index.tolist()

heatmap_data = exploded[exploded["skills_tags"].isin(top_skills)].groupby(
    ["seniority", "skills_tags"]
).size().unstack(fill_value=0)

# Reorder seniority index
order = ["Intern", "Junior", "Mid", "Senior", "Lead", "Director+"]
heatmap_data = heatmap_data.reindex([s for s in order if s in heatmap_data.index])

fig = px.imshow(
    heatmap_data, 
    labels=dict(x="Skill", y="Seniority", color="Job Count"),
    title="Top 15 Skills by Seniority Level",
    aspect="auto",
    template="plotly_dark",
    color_continuous_scale="Viridis"
)
fig.show()

## 3. Geographical Density
Where is the work? (Province-level concentration)

In [4]:
prov_counts = df["location_state"].value_counts().reset_index()
prov_counts.columns = ["province", "count"]

fig = px.pie(
    prov_counts, 
    values="count", 
    names="province", 
    title="Job Volume by Province",
    hole=0.4,
    template="plotly_dark"
)
fig.update_traces(textinfo="percent+label")
fig.show()